In [6]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import logging
from sklearn.preprocessing import StandardScaler
import copy
import random
import spectral
from skimage import color
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torchvision.transforms import Compose, Resize, ToTensor



# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Set a fixed seed for reproducibility
seed = 10
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # If using GPU

# Ensure deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

###########################################
# Helper functions
###########################################
def process_rgb(cube, bands, ill, CMFs):
    """
    Converts hyperspectral cube data to RGB using XYZ conversion.
    """
    ill_interp = np.interp(bands, ill[:, 0], ill[:, 1])
    CMFs_interp = np.column_stack([
        np.interp(bands, CMFs[:, 0], CMFs[:, 1]),
        np.interp(bands, CMFs[:, 0], CMFs[:, 2]),
        np.interp(bands, CMFs[:, 0], CMFs[:, 3])
    ])
    sp_tristREF = CMFs_interp * ill_interp[:, None]
    xyz = np.dot(cube, sp_tristREF) / np.sum(sp_tristREF[:, 1], axis=0)
    
    # Convert XYZ to RGB
    rgb = color.xyz2rgb(xyz)
    return rgb

###########################################
# 1. Load spectral data
###########################################
logging.info('Loading spectral data...')
ill = np.loadtxt('../../data/CIE_D65.txt')          
CMFs = np.loadtxt('../../data/CIE2degCMFs_1931.txt')

cube = spectral.open_image('../../data/colorChecker_SG/cubes/cubeCC_120f-ekta100-f14.hdr')
cube_ref = spectral.open_image('../../data/colorChecker_SG/cubeCC_DigitalSG_REF.hdr')

cube_data = cube.load()         
cube_ref_data = cube_ref.load()

wl_input = np.array(cube.metadata['wavelength'], dtype=float)
wl_ref   = np.array(cube_ref.metadata['wavelength'], dtype=float)

###########################################
# 2. Process RGB data
###########################################
logging.info('Processing RGB data...')
rgb_input = process_rgb(cube_data, wl_input, ill, CMFs)   
rgb_ref   = process_rgb(cube_ref_data, wl_ref, ill, CMFs)   

###########################################
# 3. Normalize data in RGB space
###########################################
logging.info('Normalizing RGB data...')

rgb_input_2d = rgb_input.reshape(-1, rgb_input.shape[-1])
rgb_ref_2d   = rgb_ref.reshape(-1, rgb_ref.shape[-1])

scaler_input = StandardScaler()
scaler_ref = StandardScaler()
X_norm = scaler_input.fit_transform(rgb_input_2d)
Y_norm = scaler_ref.fit_transform(rgb_ref_2d)

X_full = X_norm.reshape(rgb_input.shape)
Y_full = Y_norm.reshape(rgb_ref.shape)

###########################################
# 4. Prepare training data
###########################################
X_flat = X_full.reshape(-1, 3, 32, 32)  # Reshape into (N, Channels, Height, Width)
Y_flat = Y_full.reshape(-1, 3)  # Assuming 3 output labels, change as needed

n_pixels = X_flat.shape[0]
train_size = int(0.8 * n_pixels)
train_indices = np.random.choice(n_pixels, train_size, replace=False)
test_indices = np.setdiff1d(np.arange(n_pixels), train_indices)

X_train_split = X_flat[train_indices]
X_test_split  = X_flat[test_indices]
Y_train_split = Y_flat[train_indices]
Y_test_split  = Y_flat[test_indices]

# Convert to torch tensors
X_train_torch = torch.tensor(X_train_split, dtype=torch.float32)
Y_train_torch = torch.tensor(Y_train_split, dtype=torch.float32)
X_test_torch  = torch.tensor(X_test_split, dtype=torch.float32)
Y_test_torch  = torch.tensor(Y_test_split, dtype=torch.float32)

# Resize transformation to ensure the images are 32x32
resize_transform = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize to 32x32
    transforms.ToTensor()  # Make sure the data is in tensor form
])

# Apply the resize transformation (this is not needed if X_train_split and X_test_split are already 32x32)
X_train_resized = torch.stack([resize_transform(x.permute(1, 2, 0)) for x in X_train_torch])  # Permute to (H, W, C)
X_test_resized = torch.stack([resize_transform(x.permute(1, 2, 0)) for x in X_test_torch])  # Permute to (H, W, C)

# Now, reshaping them into the correct 4D format for CNN input (Batch x Channels x Height x Width)
X_train_torch_cnn = X_train_resized.view(-1, 3, 32, 32)  # 3 channels (RGB), 32x32 images
X_test_torch_cnn = X_test_resized.view(-1, 3, 32, 32)

###########################################
# 5. Define CNN model
###########################################
class SimpleCNN(nn.Module):
    def __init__(self, input_channels=3, output_size=3):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(input_channels, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 32 * 32, 128)
        self.fc2 = nn.Linear(128, output_size)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        x = x.view(x.size(0), -1)  # Flatten the tensor
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN(input_channels=3, output_size=3)

# Choose optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_function = nn.MSELoss()

# Early stopping
class EarlyStopping:
    def __init__(self, patience=10, delta=0, verbose=False, path='best_model.pth'):
        self.patience = patience  # Number of epochs to wait for improvement
        self.delta = delta  # Minimum change to qualify as an improvement
        self.verbose = verbose  # Whether to print the stop message
        self.path = path  # Path to save the best model
        self.best_loss = None  # Best validation loss
        self.counter = 0  # Counter for how many epochs without improvement
        self.early_stop = False  # Flag to stop the training
        self.best_model_wts = None  # To store the best model's weights

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model_wts = copy.deepcopy(model.state_dict())
        elif val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.best_model_wts = copy.deepcopy(model.state_dict())
            self.counter = 0  # Reset the counter
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                if self.verbose:
                    print(f"Early stopping triggered after {self.counter} epochs with no improvement.")
                    
        return self.early_stop

early_stopping = EarlyStopping(patience=50, delta=0.0001, verbose=True)

###########################################
# 6. Training loop
###########################################
epochs = 200
batch_size = 4
val_loss_min = float('inf')

logging.info('Training the model...')
for epoch in range(epochs):
    model.train()
    perm = torch.randperm(X_train_torch_cnn.size(0))
    X_train_shuffled = X_train_torch_cnn[perm]
    Y_train_shuffled = Y_train_torch[perm]
    
    for i in range(0, X_train_shuffled.size(0), batch_size):
        X_batch = X_train_shuffled[i:i+batch_size]
        Y_batch = Y_train_shuffled[i:i+batch_size]
        
        optimizer.zero_grad()
        Y_pred = model(X_batch)
        loss = loss_function(Y_pred, Y_batch)
        loss.backward()
        optimizer.step()

    # Calculate validation loss (on the test set)
    model.eval()
    with torch.no_grad():
        Y_val_pred = model(X_test_torch_cnn)
        val_loss = loss_function(Y_val_pred, Y_test_torch).item()

    # Print training and validation loss
    logging.info(f'Epoch {epoch+1}/{epochs} - Training Loss: {loss.item()} - Validation Loss: {val_loss}')

    # Check for early stopping
    early_stopping(val_loss, model)
    
    if early_stopping.early_stop:
        logging.info("Early stopping triggered.")
        break

# Load the best model weights
model.load_state_dict(early_stopping.best_model_wts)

# Evaluate the model on test data and visualize results
model.eval()
with torch.no_grad():
    Y_pred = model(X_test_torch_cnn)
    Y_pred_rgb = scaler_ref.inverse_transform(Y_pred.numpy())  # Convert back to RGB space
    Y_pred_rgb_image = Y_pred_rgb.reshape(rgb_ref.shape)

    # Visualize the results
    plt.figure(figsize=(6, 4))
    plt.imshow(Y_pred_rgb_image)
    plt.title('Predicted RGB Image')
    plt.show()



2025-03-19 15:26:07,931 - INFO - Loading spectral data...
2025-03-19 15:26:07,933 - INFO - Processing RGB data...
2025-03-19 15:26:07,934 - INFO - Normalizing RGB data...


ValueError: cannot reshape array of size 420 into shape (3,32,32)

In [ ]:
print(X_train_torch.shape)
print(X_test_torch.shape)



NameError: name 'X_train_torch' is not defined